# Sol: sistema multiagente para solicitudes de cotizacion

Prototipo inicial del sistema inteligente de SOLUCIA S.A.S. El flujo utiliza un agente supervisor y un agente asesor.

- **Agente supervisor:** recibe la solicitud y coordina el proceso.
- **Agente asesor:** analiza requisitos, identifica datos faltantes y propone una descripcion mejorada.

La demostracion funciona en modo local, sin credenciales ni llamadas externas.

## Dependencias y configuracion

Instala `Agentes Inteligentes/requirements.txt` en el entorno del notebook. El modo demo esta habilitado por defecto.

In [ ]:
from typing import TypedDict
from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph


class SolicitudState(TypedDict, total=False):
    tipo_proyecto: str
    descripcion: str
    ubicacion: str
    area_m2: float | None
    servicios: list[str]
    agente_actual: str
    decision_supervisor: str
    hechos: dict[str, str]
    preguntas_pendientes: list[str]
    descripcion_mejorada: str


def normalizar_texto(valor: object) -> str:
    return " ".join(str(valor or "").split())


def agente_supervisor(state: SolicitudState) -> SolicitudState:
    """Coordina la solicitud y la entrega al agente asesor."""
    if not normalizar_texto(state.get("descripcion")):
        raise ValueError("La solicitud debe incluir una descripcion.")
    return {
        "agente_actual": "supervisor",
        "decision_supervisor": "Enviar la solicitud al agente asesor para analizar requisitos.",
    }


def generar_descripcion_demo(hechos: dict[str, str]) -> str:
    tipo = hechos["tipo_proyecto"] or "proyecto"
    partes = [f"Se solicita asesoria para un proyecto de tipo {tipo}."]
    partes.append(
        f"La necesidad descrita por el cliente es: {hechos['descripcion_original']}"
    )
    if hechos["ubicacion"]:
        partes.append(f"El proyecto se ubica en {hechos['ubicacion']}.")
    if hechos["area_m2"]:
        partes.append(f"El area aproximada es de {hechos['area_m2']} m2.")
    if hechos["servicios"]:
        partes.append(f"Los servicios de interes son: {hechos['servicios']}.")
    return " ".join(partes)


def agente_asesor(state: SolicitudState) -> SolicitudState:
    """Analiza requisitos y prepara recomendaciones para el cliente."""
    descripcion = normalizar_texto(state.get("descripcion"))
    hechos = {
        "tipo_proyecto": normalizar_texto(state.get("tipo_proyecto")),
        "descripcion_original": descripcion,
        "ubicacion": normalizar_texto(state.get("ubicacion")),
        "area_m2": str(state.get("area_m2") or ""),
        "servicios": ", ".join(state.get("servicios", [])),
    }
    preguntas = []
    if not hechos["ubicacion"]:
        preguntas.append("Indica la ciudad o municipio del proyecto.")
    if not hechos["servicios"]:
        preguntas.append("Indica el servicio que necesitas principalmente.")
    if len(descripcion) < 40:
        preguntas.append(
            "Describe los espacios que quieres intervenir y el objetivo principal."
        )
    return {
        "agente_actual": "asesor",
        "hechos": hechos,
        "preguntas_pendientes": preguntas[:2],
        "descripcion_mejorada": generar_descripcion_demo(hechos),
    }


builder = StateGraph(SolicitudState)
builder.add_node("supervisor", agente_supervisor)
builder.add_node("asesor", agente_asesor)
builder.add_edge(START, "supervisor")
builder.add_edge("supervisor", "asesor")
builder.add_edge("asesor", END)
sol_graph = builder.compile()

display(Image(sol_graph.get_graph(xray=True).draw_mermaid_png()))

print("Grafo multiagente compilado correctamente.")

## Demostracion del flujo supervisor -> asesor

In [ ]:
entrada = {
    "tipo_proyecto": "vivienda",
    "descripcion": "Quiero remodelar mi apartamento para tener una cocina mas amplia y espacios modernos.",
    "ubicacion": "Bogota",
    "area_m2": 85,
    "servicios": ["Diseno arquitectonico", "Diseno de interiores"],
}

resultado = sol_graph.invoke(entrada)

print("1. Agente supervisor:")
print(f"   {resultado['decision_supervisor']}")
print("\n2. Agente asesor:")
print(f"   Agente activo: {resultado['agente_actual']}")
print("   Preguntas pendientes:")
if resultado["preguntas_pendientes"]:
    for pregunta in resultado["preguntas_pendientes"]:
        print(f"   - {pregunta}")
else:
    print("   - No hay preguntas pendientes.")
print("\n   Descripcion sugerida:")
print(f"   {resultado['descripcion_mejorada']}")

In [ ]:
assert resultado["agente_actual"] == "asesor"
assert resultado["decision_supervisor"]
assert resultado["descripcion_mejorada"]
assert "Bogota" in resultado["descripcion_mejorada"]
print("Prueba del flujo multiagente superada.")